# Segmentación semántica con U-Net y encoder ResNet18

Construiremos un ejemplo autocontenido que segmenta fondo, rectángulos y círculos en imágenes sintéticas.

**Conceptos:** etiqueta por píxel, encoder, decoder, skip connections, logits multiclase, entropía cruzada e IoU.

**Resultado esperado:** después de unas épocas, las máscaras predichas deberían aproximar la posición de las figuras. Los valores exactos dependen de la ejecución.


## 1. Reproducibilidad e hiperparámetros

Las imágenes sintéticas permiten entender el pipeline sin descargar anotaciones externas. Cada índice genera siempre la misma muestra gracias a una semilla local.


In [ ]:
import copy
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision import models

SEED = 42
IMAGE_SIZE = 128
NUM_CLASSES = 3
CLASS_NAMES = ["fondo", "rectángulo", "círculo"]
CLASS_COLORS = np.array(
    [
        [0, 0, 0],
        [230, 80, 80],
        [80, 220, 110],
    ],
    dtype=np.uint8,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Dispositivo:", device)


## 2. Dataset sintético

Cada imagen contiene un rectángulo rojo y un círculo verde sobre un fondo oscuro, más ruido leve. La máscara tiene enteros:

- 0 para fondo
- 1 para rectángulo
- 2 para círculo

Si las figuras se traslapan, la última figura dibujada define la clase del píxel. La salida del modelo tendrá un logit por clase y píxel.


In [ ]:
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class SyntheticShapesDataset(Dataset):
    def __init__(self, sample_count, image_size, base_seed):
        self.sample_count = sample_count
        self.image_size = image_size
        self.base_seed = base_seed

    def __len__(self):
        return self.sample_count

    def __getitem__(self, index):
        random_generator = np.random.default_rng(
            self.base_seed + index
        )

        image = Image.new(
            "RGB",
            (self.image_size, self.image_size),
            color=(20, 20, 20),
        )
        mask = Image.new(
            "L",
            (self.image_size, self.image_size),
            color=0,
        )
        image_draw = ImageDraw.Draw(image)
        mask_draw = ImageDraw.Draw(mask)

        # Rectángulo: elegimos esquina y tamaño dentro de la imagen.
        rectangle_width = int(
            random_generator.integers(24, 55)
        )
        rectangle_height = int(
            random_generator.integers(20, 50)
        )
        rectangle_x = int(
            random_generator.integers(
                5,
                self.image_size - rectangle_width - 5,
            )
        )
        rectangle_y = int(
            random_generator.integers(
                5,
                self.image_size - rectangle_height - 5,
            )
        )
        rectangle_box = [
            rectangle_x,
            rectangle_y,
            rectangle_x + rectangle_width,
            rectangle_y + rectangle_height,
        ]
        image_draw.rectangle(
            rectangle_box,
            fill=tuple(CLASS_COLORS[1]),
        )
        mask_draw.rectangle(rectangle_box, fill=1)

        # Círculo: PIL recibe la caja que contiene a la elipse.
        circle_diameter = int(
            random_generator.integers(20, 48)
        )
        circle_x = int(
            random_generator.integers(
                5,
                self.image_size - circle_diameter - 5,
            )
        )
        circle_y = int(
            random_generator.integers(
                5,
                self.image_size - circle_diameter - 5,
            )
        )
        circle_box = [
            circle_x,
            circle_y,
            circle_x + circle_diameter,
            circle_y + circle_diameter,
        ]
        image_draw.ellipse(
            circle_box,
            fill=tuple(CLASS_COLORS[2]),
        )
        mask_draw.ellipse(circle_box, fill=2)

        image_array = np.asarray(image, dtype=np.float32)
        noise = random_generator.normal(
            loc=0.0,
            scale=8.0,
            size=image_array.shape,
        )
        image_array = np.clip(
            image_array + noise,
            0,
            255,
        ).astype(np.uint8)

        image_tensor = torch.from_numpy(
            image_array.copy()
        ).permute(2, 0, 1).float() / 255.0
        image_tensor = (
            image_tensor - imagenet_mean
        ) / imagenet_std
        mask_tensor = torch.from_numpy(
            np.asarray(mask, dtype=np.int64).copy()
        )
        return image_tensor, mask_tensor


train_dataset = SyntheticShapesDataset(
    sample_count=320,
    image_size=IMAGE_SIZE,
    base_seed=SEED,
)
validation_dataset = SyntheticShapesDataset(
    sample_count=80,
    image_size=IMAGE_SIZE,
    base_seed=10_000,
)
test_dataset = SyntheticShapesDataset(
    sample_count=20,
    image_size=IMAGE_SIZE,
    base_seed=20_000,
)


Visualizamos imagen y máscara antes de construir la red. mask_to_rgb hace explícita la correspondencia clase-color.

**Resultado esperado:** las fronteras de la máscara coinciden con las figuras, incluso donde se traslapan.


In [ ]:
def denormalize_image(image_tensor):
    # Invierte la normalización de ImageNet para visualización.
    image = image_tensor.cpu() * imagenet_std + imagenet_mean
    return torch.clamp(image, 0, 1).permute(1, 2, 0).numpy()


def mask_to_rgb(mask):
    # Convierte cada índice de clase en el color definido al inicio.
    return CLASS_COLORS[np.asarray(mask, dtype=np.int64)]


sample_image, sample_mask = train_dataset[0]
figure, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(denormalize_image(sample_image))
axes[0].set_title("Imagen de entrada")
axes[1].imshow(mask_to_rgb(sample_mask.numpy()))
axes[1].set_title("Máscara objetivo")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 3. DataLoaders

El batch de imágenes tiene forma (B, 3, H, W). La máscara no tiene canal: su forma es (B, H, W) y cada valor es un índice de clase.


In [ ]:
BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

batch_images, batch_masks = next(iter(train_loader))
print("Imágenes:", batch_images.shape)
print("Máscaras:", batch_masks.shape)
print("Clases presentes:", torch.unique(batch_masks).tolist())


## 4. Arquitectura U-Net con encoder ResNet18

El encoder reduce resolución y aumenta canales. El decoder interpola y concatena features de la misma escala mediante skip connections. Así combina contexto profundo con detalle espacial.

ConvRelu representa una operación repetida sencilla: convolución 3 × 3 seguida de ReLU.


In [ ]:
def convolution_relu(in_channels, out_channels):
    # Bloque básico usado para fusionar features en el decoder.
    return nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        ),
        nn.ReLU(inplace=True),
    )


class ResNet18UNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        self.encoder_stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
        )
        self.encoder_pool = backbone.maxpool
        self.encoder_layer_1 = backbone.layer1
        self.encoder_layer_2 = backbone.layer2
        self.encoder_layer_3 = backbone.layer3
        self.encoder_layer_4 = backbone.layer4

        self.decode_3 = convolution_relu(512 + 256, 256)
        self.decode_2 = convolution_relu(256 + 128, 128)
        self.decode_1 = convolution_relu(128 + 64, 64)
        self.decode_0 = convolution_relu(64 + 64, 64)

        self.input_skip = convolution_relu(3, 32)
        self.decode_input = convolution_relu(64 + 32, 32)
        self.classifier = nn.Conv2d(
            32,
            num_classes,
            kernel_size=1,
        )

    def forward(self, images):
        input_features = self.input_skip(images)

        skip_0 = self.encoder_stem(images)
        skip_1 = self.encoder_layer_1(
            self.encoder_pool(skip_0)
        )
        skip_2 = self.encoder_layer_2(skip_1)
        skip_3 = self.encoder_layer_3(skip_2)
        bottleneck = self.encoder_layer_4(skip_3)

        decoded = F.interpolate(
            bottleneck,
            size=skip_3.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        decoded = self.decode_3(
            torch.cat([decoded, skip_3], dim=1)
        )

        decoded = F.interpolate(
            decoded,
            size=skip_2.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        decoded = self.decode_2(
            torch.cat([decoded, skip_2], dim=1)
        )

        decoded = F.interpolate(
            decoded,
            size=skip_1.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        decoded = self.decode_1(
            torch.cat([decoded, skip_1], dim=1)
        )

        decoded = F.interpolate(
            decoded,
            size=skip_0.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        decoded = self.decode_0(
            torch.cat([decoded, skip_0], dim=1)
        )

        decoded = F.interpolate(
            decoded,
            size=images.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        decoded = self.decode_input(
            torch.cat([decoded, input_features], dim=1)
        )
        return self.classifier(decoded)


Verificamos formas antes de entrenar. Para B imágenes y tres clases, los logits deben tener forma (B, 3, 128, 128), igualando alto y ancho de la máscara.

**Resultado esperado:** la aserción termina sin error.


In [ ]:
model = ResNet18UNet(num_classes=NUM_CLASSES).to(device)

with torch.inference_mode():
    example_logits = model(batch_images[:2].to(device))

print("Entrada:", batch_images[:2].shape)
print("Logits:", example_logits.shape)
print("Objetivo:", batch_masks[:2].shape)

assert example_logits.shape == (
    2,
    NUM_CLASSES,
    IMAGE_SIZE,
    IMAGE_SIZE,
)


## 5. Pérdida e IoU

CrossEntropyLoss compara los tres logits de cada píxel con su clase entera; no aplicamos softmax dentro del modelo.

Para una clase:

IoU = píxeles de intersección / píxeles de unión

Reportamos mean IoU de rectángulo y círculo, excluyendo fondo para que una gran región vacía no domine la interpretación.


In [ ]:
criterion = nn.CrossEntropyLoss()

def run_segmentation_epoch(
    model,
    data_loader,
    criterion,
    device,
    optimizer=None,
):
    is_training = optimizer is not None
    model.train(mode=is_training)

    total_loss = 0.0
    total_images = 0
    intersections = np.zeros(NUM_CLASSES, dtype=np.float64)
    unions = np.zeros(NUM_CLASSES, dtype=np.float64)

    for images, masks in data_loader:
        images = images.to(device)
        masks = masks.to(device)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, masks)
            if is_training:
                loss.backward()
                optimizer.step()

        predictions = logits.argmax(dim=1)
        for class_index in range(1, NUM_CLASSES):
            predicted_class = predictions == class_index
            target_class = masks == class_index
            intersections[class_index] += (
                predicted_class & target_class
            ).sum().item()
            unions[class_index] += (
                predicted_class | target_class
            ).sum().item()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    class_ious = [
        intersections[class_index] / unions[class_index]
        for class_index in range(1, NUM_CLASSES)
        if unions[class_index] > 0
    ]
    return {
        "loss": total_loss / total_images,
        "mean_iou": float(np.mean(class_ious)),
    }


## 6. Entrenamiento y checkpoint

Entrenamos encoder y decoder con AdamW. Para un equipo sin GPU se puede reducir EPOCHS o el número de muestras, manteniendo el mismo pipeline.

Elegimos el checkpoint con mayor IoU de validación y guardamos configuración junto a los pesos.

**Resultado esperado:** IoU aumenta desde una predicción inicialmente pobre. Si no ocurre, inspeccione primero imágenes, máscaras y shapes.


In [ ]:
LEARNING_RATE = 1e-4
EPOCHS = 5
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

best_validation_iou = -1.0
best_state = copy.deepcopy(model.state_dict())
history = {
    "train_loss": [],
    "validation_loss": [],
    "train_iou": [],
    "validation_iou": [],
}

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_metrics = run_segmentation_epoch(
        model,
        train_loader,
        criterion,
        device,
        optimizer,
    )
    validation_metrics = run_segmentation_epoch(
        model,
        validation_loader,
        criterion,
        device,
    )

    history["train_loss"].append(train_metrics["loss"])
    history["validation_loss"].append(validation_metrics["loss"])
    history["train_iou"].append(train_metrics["mean_iou"])
    history["validation_iou"].append(
        validation_metrics["mean_iou"]
    )

    if validation_metrics["mean_iou"] > best_validation_iou:
        best_validation_iou = validation_metrics["mean_iou"]
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Época {epoch:02d}/{EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f}, "
        f"IoU {train_metrics['mean_iou']:.3f} | "
        f"val loss {validation_metrics['loss']:.4f}, "
        f"IoU {validation_metrics['mean_iou']:.3f} | "
        f"{time.time() - start_time:.1f} s"
    )

model.load_state_dict(best_state)

output_directory = Path(
    "outputs/unet_resnet18_synthetic"
)
output_directory.mkdir(parents=True, exist_ok=True)
torch.save(
    model.state_dict(),
    output_directory / "best_model.pt",
)

experiment_config = {
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "classes": CLASS_NAMES,
    "train_samples": len(train_dataset),
    "validation_samples": len(validation_dataset),
    "test_samples": len(test_dataset),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "encoder_weights": "ResNet18_Weights.DEFAULT",
}
with (output_directory / "course_experiment.json").open(
    "w",
    encoding="utf-8",
) as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Mejor IoU de validación:", f"{best_validation_iou:.3f}")
print("Resultados guardados en:", output_directory)


In [ ]:
epoch_numbers = range(1, EPOCHS + 1)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epoch_numbers, history["train_loss"], label="Train")
axes[0].plot(
    epoch_numbers,
    history["validation_loss"],
    label="Validación",
)
axes[0].set_title("Pérdida")
axes[0].set_xlabel("Época")
axes[0].legend()

axes[1].plot(epoch_numbers, history["train_iou"], label="Train")
axes[1].plot(
    epoch_numbers,
    history["validation_iou"],
    label="Validación",
)
axes[1].set_title("Mean IoU sin fondo")
axes[1].set_xlabel("Época")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Evaluación y visualización

Test se evalúa una sola vez después de elegir el checkpoint. Mostramos entrada, máscara real y máscara predicha.

**Resultado esperado:** las regiones predichas deben coincidir aproximadamente con forma y posición. Los bordes suelen ser la zona más difícil.


In [ ]:
test_metrics = run_segmentation_epoch(
    model,
    test_loader,
    criterion,
    device,
)
print(
    f"Test loss: {test_metrics['loss']:.4f} | "
    f"Test mean IoU: {test_metrics['mean_iou']:.3f}"
)

test_images, test_masks = next(iter(test_loader))
model.eval()
with torch.inference_mode():
    test_predictions = model(
        test_images.to(device)
    ).argmax(dim=1).cpu()

figure, axes = plt.subplots(4, 3, figsize=(10, 13))
for row_index in range(4):
    axes[row_index, 0].imshow(
        denormalize_image(test_images[row_index])
    )
    axes[row_index, 1].imshow(
        mask_to_rgb(test_masks[row_index].numpy())
    )
    axes[row_index, 2].imshow(
        mask_to_rgb(test_predictions[row_index].numpy())
    )

    for axis in axes[row_index]:
        axis.axis("off")

axes[0, 0].set_title("Entrada")
axes[0, 1].set_title("Ground truth")
axes[0, 2].set_title("Predicción")
plt.tight_layout()
plt.show()


## 8. Ejercicios

**Ejercicio 1 — Sin skip connections.** Sustituya una concatenación por solo la feature del decoder y ajuste canales.

**Resultado esperado:** puede perderse precisión de bordes porque el decoder recibe menos detalle espacial.

**Ejercicio 2 — Desbalance.** Reduzca el tamaño de las figuras. Compare accuracy de píxeles e IoU sin fondo.

**Resultado esperado:** accuracy puede parecer alta al predecir fondo; IoU de objetos revela el problema.

**Ejercicio 3 — Más clases.** Agregue triángulos con etiqueta 3. Actualice NUM_CLASSES, colores y generación.

**Resultado esperado:** la salida cambia a cuatro canales, mientras la máscara conserva forma (B, H, W) con valores 0–3.

**Checklist:** entradas normalizadas, máscaras enteras, logits sin softmax para CrossEntropyLoss, shapes compatibles, validación para selección y test para reporte final.
